In [0]:
# COMMAND ----------
# SILVER LAYER: ADVANCED FEATURE ENGINEERING & HIPAA COMPLIANCE
# Target: workspace.healthcare2.silver_patients
# Use: Transform Bronze raw data into standardized metrics for Gold Star Schema
# COMMAND ----------

from pyspark.sql.functions import sha2, col, when, current_timestamp, round, datediff, split, size, lit

# 1. Load from Bronze Layer
# Reading the raw data ingested in the bronze layer
bronze_df = spark.read.table("workspace.healthcare3.bronze_patients")

# 2. Transformation & Feature Engineering Logic
silver_df = (bronze_df
    # --- SECURITY: HIPAA Compliance ---
    # Hashes Patient IDs to protect PII (Personally Identifiable Information)
    .withColumn("patient_key", sha2(col("patient_id").cast("string"), 256))
    
    # --- CFO: Finance & Cashback ROI (Personalized Priority) ---
    # Recovery Rate: Percentage of the insurance claim actually recovered as cash
    .withColumn("cashback_recovery_rate", 
        round(col("approved_claim_amount") / col("claim_amount"), 4))
    # Revenue Leakage: The dollar value lost due to denials or underpayment
    .withColumn("revenue_leakage", 
        round(col("claim_amount") - col("approved_claim_amount"), 2))
    # Casting raw financial field for downstream Gold aggregation
    .withColumn("total_billed_amount", col("total_billed_amount").cast("double"))
    
    # --- COO: Operational Efficiency ---
    # Length of Stay (LOS): Days between admission and discharge
    .withColumn("los_days", datediff(col("discharge_date"), col("admission_date")))
    # ER Overload Flag: Identifies efficiency bottlenecks (Wait time > 3 hours)
    .withColumn("is_er_overload", 
        when((col("dept_name") == 'ER') & (col("wait_time_min") > 180), 1).otherwise(0))
    # Pass through raw operational indicators
    .withColumn("is_emergency_admission", col("is_emergency_admission").cast("int"))
    
    # --- CMO: Clinical Quality & Risk ---
    # Risk Segment: Categorizes patients based on vitals and lab criticalities
    .withColumn("clinical_risk_segment", 
        when((col("blood_pressure_status") == 'High') | (col("lab_result_status") == 'Critical'), "High Risk")
        .otherwise("Stable"))
    # Comorbidity Count: Parses the ICD-10 history string to count conditions
    .withColumn("comorbidity_count", 
        when(col("icd10_history").isNull(), 0)
        .otherwise(size(split(col("icd10_history"), ","))))
    
    # --- METADATA ---
    .withColumn("silver_load_timestamp", current_timestamp())
)

# 3. Final Schema Selection
# We select all original features + new engineered features required for the Gold Fact/Dims
final_silver_df = silver_df.select(
    "patient_key", "admission_date", "discharge_date", "age", "gender", 
    "blood_group", "treatment_type", "blood_pressure_status", "heart_rate", 
    "oxygen_saturation", "lab_result_status", "dept_name", "wait_time_min", 
    "bed_occupancy_status", "is_emergency_admission", "staff_ratio", 
    "payer_name", "claim_status", "denial_reason", "total_billed_amount", 
    "claim_amount", "approved_claim_amount", "cashback_recovery_rate", 
    "revenue_leakage", "los_days", "is_er_overload", 
    "clinical_risk_segment", "comorbidity_count", "silver_load_timestamp"
)

# 4. Save to Silver Table (Delta Format)
# Optimized with overwriteSchema to handle updates to the feature set
(final_silver_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("workspace.healthcare3.silver_patients"))

# 5. Data Quality Audit
print("✅ Success: Silver Layer Transformation Complete.")

✅ Success: Silver Layer Transformation Complete.
